# Notebook 06 · revisión 01: joins, ventanas y ejecución observable de Spark
**Curso de Postgrado: BigData, Especialización en Bases de datos**  
Preparado para la clase de Esteban Hernández, PhD · CyberColombia.

Esta revisión conserva los cálculos y controles del notebook 06 y añade observación de particiones,
tareas concurrentes, agregación parcial, shuffle y uso de disco. No sustituye el notebook 05.

**Entrada:** productos de 05 en `kit/salidas/E07`.  
**Salida de esta revisión:** `kit/salidas/E07_rev01`. Los productos originales se conservan.

## Qué cambia en rev 01
| Adición o ajuste | Qué debe poder explicar el estudiante |
|---|---|
| Configuración observable y event logs | Qué recursos tiene la aplicación y dónde registra su ejecución |
| Ejemplo pequeño de agregación | Sumas por clave dentro de particiones y combinación final |
| Sonda con ocho particiones | Cuándo se solapan tareas en el mismo entorno local |
| Inventarios de temporales | Diferencia entre bloques de shuffle, temporales y salidas Parquet |
| Métricas por etapa y tarea | Diferencia entre Shuffle Write y Spill (Disk/Memory) |
| Dos experimentos de particionado | Diferencia entre observar AQE y fijar un experimento sin AQE |
| Cierre separado | Capturar evidencia antes de `spark.stop()` y conservar los event logs |

## Preparación y alcance
Usar Python 3.12, PySpark **4.0.1**, JDK 21 y el kernel del `.venv` del curso.
Preparar el entorno según [WSL](../../docs/instalacion-wsl.md) o
[Linux/macOS](../../docs/instalacion-linux-macos.md), sin crear otro entorno para este notebook.
Completar [05 Spark](05_Spark_lectura_y_agregacion.ipynb) y **detener su sesión** antes de comenzar.

El 05 prepara los CSV completos con Pandas. Esta fase requiere memoria local y no demuestra una
preparación distribuida de TB. Aquí Spark lee los Parquet ya preparados. El uso de `local[2]`
permite concurrencia en una máquina; no demuestra red entre servidores ni varios executors remotos.
Las UDF Python no forman parte de los cálculos de este taller: se usan expresiones nativas de Spark.

## 0. Configuración, recursos y rutas
Una **partición** organiza datos; una **tarea** procesa una partición para una etapa;
un **executor** aporta un proceso y recursos donde se ejecutan tareas.
El driver planifica y programa trabajo. Los executors procesan y transfieren bloques.

`local[2]` permite hasta dos tareas simultáneas con una CPU por tarea. No crea dos servidores.
`spark.sql.shuffle.partitions=4` configura ciertos shuffles SQL: no fija todas las particiones
para toda la aplicación. La lectura, `repartition` y AQE pueden producir cantidades distintas.

La configuración del directorio local y los event logs debe aplicarse **antes** de crear el contexto.
Si existe otro contexto, detenerlo desde su notebook o reiniciar el kernel.

In [1]:
from pathlib import Path
import os, sys, json, io, contextlib, time
from datetime import datetime, timezone
from urllib.request import build_opener, ProxyHandler
from urllib.parse import quote
import pandas as pd
import duckdb
from IPython.display import display
from pyspark import SparkContext
from pyspark.sql import SparkSession, functions as F

# Ajustar solo si Jupyter se inició fuera del repositorio.
REPOSITORIO = None  # Ejemplo: Path("/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate")
if REPOSITORIO is None:
    candidatos = [Path.cwd(), *Path.cwd().parents]
    REPOSITORIO = next((p for p in candidatos if (p / "kit/requirements_spark.txt").is_file()), None)
assert REPOSITORIO is not None, "Asigna REPOSITORIO a la raíz de tu clon."
KIT = Path(REPOSITORIO).expanduser().resolve() / "kit"
ENTRADA = KIT / "salidas/E07"
OUT = KIT / "salidas/E07_rev01"
for nombre in ["eva.parquet", "dim_municipio.parquet", "spark_rendimiento", "rendimiento_duckdb.parquet"]:
    assert (ENTRADA / nombre).exists(), f"Completa 05: falta {ENTRADA / nombre}"
assert SparkContext._active_spark_context is None, "Detén la sesión anterior o reinicia el kernel."
for carpeta in [OUT, OUT / "temporales_spark", OUT / "event_logs", OUT / "evidencias"]:
    carpeta.mkdir(parents=True, exist_ok=True)
os.environ["PYSPARK_PYTHON"] = sys.executable
spark = (SparkSession.builder.master("local[2]").appName("Curso-BigData-E07-06-rev01")
    .config("spark.task.cpus", "1")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.local.dir", str(OUT / "temporales_spark"))
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", (OUT / "event_logs").as_uri())
    .config("spark.eventLog.compress", "false")
    .config("spark.eventLog.rolling.enabled", "false")
    .getOrCreate())
sc = spark.sparkContext
assert spark.version == "4.0.1", f"Instala kit/requirements_spark.txt; versión actual: {spark.version}"
print("Python:", sys.executable, "Spark:", spark.version)
print("Master:", sc.master, "CPU por tarea:", sc.getConf().get("spark.task.cpus"))
print("Paralelismo predeterminado:", sc.defaultParallelism)
print("UI activa:", sc.uiWebUrl, "Aplicación:", sc.applicationId)
print("Shuffle inicial:", spark.conf.get("spark.sql.shuffle.partitions"))
print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Entrada:", ENTRADA, "Salida:", OUT)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/17 17:05:26 WARN Utils: Your hostname, POS-316-A396596, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/17 17:05:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/17 17:05:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/17 17:05:29 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


Python: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv/bin/python Spark: 4.0.1
Master: local[2] CPU por tarea: 1
Paralelismo predeterminado: 2
UI activa: http://10.255.255.254:4040 Aplicación: local-1792274733986
Shuffle inicial: 4
AQE: true
Entrada: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07 Salida: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07_rev01


### Herramientas de observación
Las siguientes funciones consultan la API REST de la UI **de esta aplicación local**, guardan planes
y crean inventarios de archivos. No ejecutan consultas agrícolas ocultas.
Los errores de observación quedan registrados: ausencia de métricas no se convierte en cero.

Un inventario contiene nombres y tamaños de archivos existentes en ese momento. No demuestra que
sean archivos de spill ni permite recuperar temporales ya eliminados. Los event logs conservan
los eventos y métricas para su revisión posterior, incluso cuando la UI deja de estar disponible.

In [2]:
HTTP_LOCAL = build_opener(ProxyHandler({}))  # Evitar enviar la UI local a un proxy externo.
API = f"{sc.uiWebUrl}/api/v1/applications/{quote(sc.applicationId, safe='')}"

def guardar_json(nombre, datos):
    destino = OUT / "evidencias" / nombre
    destino.write_text(json.dumps(datos, ensure_ascii=False, indent=2), encoding="utf-8")
    return destino

def api_local(recurso):
    with HTTP_LOCAL.open(API + recurso, timeout=15) as respuesta:
        return json.load(respuesta)

def guardar_plan(df, nombre):
    captura = io.StringIO()
    with contextlib.redirect_stdout(captura):
        df.explain("formatted")
    texto = captura.getvalue()
    (OUT / "evidencias" / nombre).write_text(texto, encoding="utf-8")
    print(texto)
    return texto

# API interna de PySpark clásico/Spark 4.0.1; no disponible así en Spark Connect.
# En local corresponde al entorno de ejecución. En clúster esto inspeccionaría el driver.
disk_manager = sc._jsc.sc().env().blockManager().diskBlockManager()
directorios_reales = [Path(d.getAbsolutePath()) for d in disk_manager.localDirs()]
print("Configurado:", sc.getConf().get("spark.local.dir", "(predeterminado)"))
print("SPARK_LOCAL_DIRS:", os.environ.get("SPARK_LOCAL_DIRS"))
print("LOCAL_DIRS:", os.environ.get("LOCAL_DIRS"))
print("Directorios efectivos:", [str(p) for p in directorios_reales])

def inventario(etiqueta):
    filas = []
    for raiz in directorios_reales:
        for archivo in raiz.rglob("*"):
            try:
                if archivo.is_file():
                    st = archivo.stat()
                    filas.append({"ruta": str(archivo), "bytes": st.st_size,
                                  "modificado_ns": st.st_mtime_ns})
            except FileNotFoundError:
                pass  # El executor puede eliminar un temporal mientras inspeccionamos.
    guardar_json(f"inventario_{etiqueta}.json", filas)
    print(etiqueta, "archivos presentes:", len(filas), "bytes presentes:", sum(f["bytes"] for f in filas))
    display(pd.DataFrame(filas).head(20))
    return filas

def capturar_metricas(etiqueta):
    try:
        etapas = api_local("/stages")
        executors = api_local("/executors")
        trabajos = api_local("/jobs")
        guardar_json(f"metricas_{etiqueta}.json", {"etapas": etapas, "executors": executors, "jobs": trabajos})
        columnas = ["stageId", "attemptId", "status", "numTasks", "numCompleteTasks",
                    "shuffleReadBytes", "shuffleWriteBytes", "memoryBytesSpilled", "diskBytesSpilled"]
        tabla = pd.DataFrame(etapas).reindex(columns=columnas)
        tabla.to_csv(OUT / "evidencias" / f"etapas_{etiqueta}.csv", index=False)
        display(tabla)
        return etapas
    except Exception as error:
        guardar_json(f"error_metricas_{etiqueta}.json", {"tipo": type(error).__name__, "mensaje": str(error)})
        print("No se pudieron capturar métricas; revisar la UI y los event logs:", error)
        return None

inventario_inicial = inventario("inicio")
try:
    executors_iniciales = api_local("/executors")
    display(pd.DataFrame(executors_iniciales).reindex(columns=["id", "hostPort", "totalCores", "maxTasks", "activeTasks"]))
except Exception as error:
    executors_iniciales = None
    print("Revisar pestaña Executors manualmente:", error)

Configurado: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07_rev01/temporales_spark
SPARK_LOCAL_DIRS: None
LOCAL_DIRS: None
Directorios efectivos: ['/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07_rev01/temporales_spark/blockmgr-4e75f611-e93a-49c5-8ce4-abe50d5393d1']
inicio archivos presentes: 0 bytes presentes: 0


""


Revisar pestaña Executors manualmente: timed out


## A. Sumas parciales: por tarea sobre una partición
Usamos **seis filas didácticas**, distintas del conjunto EVA. Las repartimos y persistimos para que
la distribución inspeccionada se mantenga durante este ejemplo pequeño.

La tabla de sumas por partición se reconstruye con Pandas a partir de las seis filas observadas.
**Es una explicación del algoritmo, no una lectura de los buffers internos de HashAggregate.**
El plan de la consulta nativa permite buscar agregación parcial, Exchange y agregación final.
Un executor puede ejecutar muchas tareas; no existe una única suma parcial por executor.

In [3]:
filas_demo = [("A", 10.0, 2.0), ("A", 6.0, 3.0), ("A", 9.0, 3.0),
              ("A", 15.0, 2.0), ("B", 6.0, 2.0), ("B", 9.0, 3.0)]
entrada_demo = spark.createDataFrame(filas_demo, ["clave", "produccion_t", "area_ha"]).repartition(2).persist()
entrada_demo.count()  # Materializa únicamente estas seis filas.
distribucion = entrada_demo.withColumn("particion_entrada", F.spark_partition_id()).toPandas()
display(distribucion.sort_values(["particion_entrada", "clave"]))
parciales_didacticos = (distribucion.groupby(["particion_entrada", "clave"], as_index=False)
                       [["produccion_t", "area_ha"]].sum())
display(parciales_didacticos)
parciales_didacticos.to_csv(OUT / "evidencias/sumas_por_particion_demo.csv", index=False)
resultado_demo = (entrada_demo.groupBy("clave")
    .agg(F.sum("produccion_t").alias("produccion_t"), F.sum("area_ha").alias("area_ha"))
    .withColumn("rendimiento_t_ha", F.col("produccion_t") / F.col("area_ha")))
plan_demo_antes = guardar_plan(resultado_demo, "demo_plan_antes.txt")
sc.setJobGroup("REV01_DEMO_AGREGADO", "Ejemplo pequeño: sumas parciales y finales")
try:
    filas_resultado_demo = resultado_demo.collect()  # Solo dos claves.
finally:
    sc.setLocalProperty("spark.jobGroup.id", None)
    sc.setLocalProperty("spark.job.description", None)
plan_demo_despues = guardar_plan(resultado_demo, "demo_plan_despues.txt")
demo_pd = pd.DataFrame([r.asDict() for r in filas_resultado_demo]).sort_values("clave").reset_index(drop=True)
reconstruido = parciales_didacticos.groupby("clave", as_index=False)[["produccion_t", "area_ha"]].sum()
reconstruido["rendimiento_t_ha"] = reconstruido.produccion_t / reconstruido.area_ha
pd.testing.assert_frame_equal(demo_pd[reconstruido.columns], reconstruido, check_dtype=False)
assert demo_pd.set_index("clave").loc["A", "rendimiento_t_ha"] == 4.0
assert demo_pd.set_index("clave").loc["B", "rendimiento_t_ha"] == 3.0
display(demo_pd)
entrada_demo.unpersist()

,clave,produccion_t,area_ha,particion_entrada
0,A,10.0,2.0,0
1,A,15.0,2.0,0
2,A,9.0,3.0,1
3,A,6.0,3.0,1
4,B,9.0,3.0,1
5,B,6.0,2.0,1


,particion_entrada,clave,produccion_t,area_ha
0,0,A,25.0,4.0
1,1,A,15.0,6.0
2,1,B,15.0,5.0


== Physical Plan ==
AdaptiveSparkPlan (13)
+- Project (12)
   +- HashAggregate (11)
      +- Exchange (10)
         +- HashAggregate (9)
            +- InMemoryTableScan (1)
                  +- InMemoryRelation (2)
                        +- AdaptiveSparkPlan (8)
                           +- == Final Plan ==
                              ResultQueryStage (6)
                              +- ShuffleQueryStage (5), Statistics(sizeInBytes=240.0 B, rowCount=6)
                                 +- Exchange (4)
                                    +- * Scan ExistingRDD (3)
                           +- == Initial Plan ==
                              Exchange (7)
                              +- Scan ExistingRDD (3)


(1) InMemoryTableScan
Output [3]: [clave#0, produccion_t#1, area_ha#2]
Arguments: [clave#0, produccion_t#1, area_ha#2]

(2) InMemoryRelation
Arguments: [clave#0, produccion_t#1, area_ha#2], StorageLevel(disk, memory, deserialized, 1 replicas)

(3) Scan ExistingRDD [codegen id :

,clave,produccion_t,area_ha,rendimiento_t_ha
0,A,40.0,10.0,4.0
1,B,15.0,5.0,3.0


DataFrame[clave: string, produccion_t: double, area_ha: double]

**Interpretación del recorrido:** cada tarea produce acumulados por clave dentro de su partición.
El shuffle encamina los acumulados a particiones de consumo. Una partición de consumo puede incluir
muchas claves. Su tarea combina los parciales en el executor al que se asigne.

En el shuffle sort-based habitual hay buffers en memoria y archivos de bloques en disco local.
La lectura de un bloque remoto utiliza red; en este taller local no hay transferencia entre servidores.
Si falta memoria, los operadores compatibles pueden escribir spill adicional.
El resultado posterior puede alimentar otro operador, persistirse o escribirse: no queda asociado
permanentemente a un executor. Con AQE, comparar el plan inicial y final de esta acción pequeña.

**Pregunta:** ¿el número de claves de salida tiene que coincidir con el número de particiones?

## B. Observar ocho tareas de entrada en `local[2]`
Este ensayo usa un millón de números generados por `spark.range`: **datos sintéticos de diagnóstico**,
no nuevas observaciones agrícolas. El cálculo utiliza funciones nativas de Spark y no una UDF Python.
La columna `particion_entrada` identifica de dónde proceden las filas antes de agregar.

La lectura tiene ocho particiones. Con `local[2]` y una CPU por tarea hay capacidad para dos tareas
simultáneas, pero ocho particiones no equivalen a ocho tareas ejecutándose al mismo tiempo.
El ensayo no promete una aceleración ni exige observar exactamente dos tareas solapadas:
se registra lo que efectivamente ocurrió. El arranque, la duración y los recursos influyen.

In [4]:
sc.setJobGroup("REV01_CONCURRENCIA", "Ocho particiones; operaciones nativas; dos hilos locales")
try:
    sonda = spark.range(0, 1_000_000, numPartitions=8)
    print("Particiones de entrada de la sonda:", sonda.rdd.getNumPartitions())
    consulta_sonda = (sonda.select(F.spark_partition_id().alias("particion_entrada"),
                                  F.sin(F.col("id").cast("double")).alias("senal"))
        .groupBy("particion_entrada").agg(F.count("*").alias("filas"), F.sum("senal").alias("suma_senal")))
    sonda_pd = pd.DataFrame([r.asDict() for r in consulta_sonda.collect()])
finally:
    sc.setLocalProperty("spark.jobGroup.id", None)
    sc.setLocalProperty("spark.job.description", None)
assert sonda_pd.filas.sum() == 1_000_000
assert len(sonda_pd) == 8
display(sonda_pd.sort_values("particion_entrada"))
sonda_pd.to_csv(OUT / "evidencias/sonda_particiones.csv", index=False)
plan_sonda = guardar_plan(consulta_sonda, "sonda_plan_despues.txt")

Particiones de entrada de la sonda: 8


,particion_entrada,filas,suma_senal
0,0,125000,1.163859
1,1,125000,0.331094
2,2,125000,-1.610708
3,3,125000,1.842739
4,4,125000,-0.876277
5,5,125000,-0.660104
6,6,125000,1.767162
7,7,125000,-1.724881


== Physical Plan ==
AdaptiveSparkPlan (11)
+- == Final Plan ==
   ResultQueryStage (8)
   +- * HashAggregate (7)
      +- AQEShuffleRead (6)
         +- ShuffleQueryStage (5), Statistics(sizeInBytes=256.0 B, rowCount=8)
            +- Exchange (4)
               +- * HashAggregate (3)
                  +- * Project (2)
                     +- * Range (1)
+- == Initial Plan ==
   HashAggregate (10)
   +- Exchange (9)
      +- HashAggregate (3)
         +- Project (2)
            +- Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#250L]
Arguments: Range (0, 1000000, step=1, splits=Some(8))

(2) Project [codegen id : 1]
Output [2]: [SPARK_PARTITION_ID() AS particion_entrada#251, SIN(cast(id#250L as double)) AS senal#252]
Input [1]: [id#250L]

(3) HashAggregate [codegen id : 1]
Input [2]: [particion_entrada#251, senal#252]
Keys [1]: [particion_entrada#251]
Functions [2]: [partial_count(1), partial_sum(senal#252)]
Aggregate Attributes [2]: [count#259L, sum#260]
Results [3]: [particion

In [5]:
# Consultar solo las etapas asociadas al grupo del ensayo; no toda la aplicación.
concurrencia_observada = None
tareas_sonda = []
try:
    jobs_sonda = [j for j in api_local("/jobs") if j.get("jobGroup") == "REV01_CONCURRENCIA"]
    ids_etapas = {i for j in jobs_sonda for i in j["stageIds"]}
    etapas_sonda = [e for e in api_local("/stages") if e["stageId"] in ids_etapas and e["status"] == "COMPLETE"]
    for etapa in etapas_sonda:
        sid, intento = etapa["stageId"], etapa["attemptId"]
        detalle = api_local(f"/stages/{sid}/{intento}/taskList?length=10000")
        for tarea in detalle:
            tareas_sonda.append({"stageId": sid, "stageAttemptId": intento, **tarea})
    guardar_json("tareas_sonda.json", tareas_sonda)
    filas_tareas = []
    for t in tareas_sonda:
        if t.get("duration") is not None and t.get("status") == "SUCCESS":
            inicio = pd.Timestamp(t["launchTime"])
            filas_tareas.append({"stageId": t["stageId"], "stageAttemptId": t["stageAttemptId"],
                "taskId": t["taskId"], "particion": t.get("partitionId", t["index"]),
                "executorId": t["executorId"], "inicio": inicio,
                "fin": inicio + pd.Timedelta(milliseconds=t["duration"])})
    tabla_tareas = pd.DataFrame(filas_tareas)
    if not tabla_tareas.empty:
        display(tabla_tareas.sort_values(["inicio", "taskId"]))
        tabla_tareas.to_csv(OUT / "evidencias/tareas_sonda.csv", index=False)
        resumen_solape = []
        capacidad_ui = {e["id"]: e.get("maxTasks") for e in api_local("/executors")}
        umbral_ms = 10.0  # Excluir solapes mínimos de notificación; no estimar CPU ocupadas.
        for executor, grupo in tabla_tareas.groupby("executorId"):
            pares = []
            for _, etapa in grupo.groupby(["stageId", "stageAttemptId"]):
                registros = list(etapa.itertuples())
                for i, tarea_a in enumerate(registros):
                    for tarea_b in registros[i + 1:]:
                        ms = (min(tarea_a.fin, tarea_b.fin) - max(tarea_a.inicio, tarea_b.inicio)).total_seconds() * 1000
                        if ms >= umbral_ms:
                            pares.append({"stageId": int(tarea_a.stageId), "tarea_a": int(tarea_a.taskId),
                                          "tarea_b": int(tarea_b.taskId), "solapamiento_ms": round(ms, 3)})
            mejor = max(pares, key=lambda x: x["solapamiento_ms"]) if pares else None
            resumen_solape.append({"executorId": executor, "capacidad_tareas_reportada_UI": capacidad_ui.get(executor),
                                   "umbral_ms": umbral_ms, "par_con_solape_claro": mejor})
        concurrencia_observada = resumen_solape
        guardar_json("concurrencia_observada.json", resumen_solape)
        display(pd.DataFrame(resumen_solape))
    else:
        print("No hay tareas completas recuperadas; revisar el timeline de Stages en la UI.")
except Exception as error:
    guardar_json("error_sonda.json", {"tipo": type(error).__name__, "mensaje": str(error)})
    print("No se obtuvo evidencia automática de solapamiento:", error)

,stageId,stageAttemptId,taskId,particion,executorId,inicio,fin
1,15,0,12,0,driver,2026-10-17 22:06:26.032000+00:00,2026-10-17 22:06:26.362000+00:00
2,15,0,13,1,driver,2026-10-17 22:06:26.034000+00:00,2026-10-17 22:06:26.299000+00:00
3,15,0,14,2,driver,2026-10-17 22:06:26.298000+00:00,2026-10-17 22:06:26.452000+00:00
4,15,0,15,3,driver,2026-10-17 22:06:26.361000+00:00,2026-10-17 22:06:26.508000+00:00
5,15,0,16,4,driver,2026-10-17 22:06:26.451000+00:00,2026-10-17 22:06:26.580000+00:00
6,15,0,17,5,driver,2026-10-17 22:06:26.507000+00:00,2026-10-17 22:06:26.654000+00:00
7,15,0,18,6,driver,2026-10-17 22:06:26.579000+00:00,2026-10-17 22:06:26.746000+00:00
8,15,0,19,7,driver,2026-10-17 22:06:26.652000+00:00,2026-10-17 22:06:26.859000+00:00
0,17,0,20,0,driver,2026-10-17 22:06:26.955000+00:00,2026-10-17 22:06:27.117000+00:00


,executorId,capacidad_tareas_reportada_UI,umbral_ms,par_con_solape_claro
0,driver,2,10.0,"{'stageId': 15, 'tarea_a': 12, 'tarea_b': 13, ..."


El solapamiento se calcula a partir de tiempos de lanzamiento y duración de tareas completas.
Se muestra un par con al menos 10 ms de solapamiento, si existe, separado de la capacidad
`maxTasks` que informa la UI. No se deduce el máximo de tareas activas ni CPU ocupadas sumando
estos intervalos: las notificaciones asíncronas pueden generar solapes mínimos entre tareas
que se relevan en el mismo recurso. No inferir número de máquinas a partir del identificador
`driver`, que puede aparecer como executor en modo local. Si no aparece un par con solapamiento claro, documentar la ausencia de evidencia y revisar
el timeline; no inventar una medición de dos tareas simultáneas.

En un clúster convencional, un executor con cuatro CPU asignadas y `spark.task.cpus=1` puede
alojar hasta cuatro tareas concurrentes, sujeto a recursos adicionales y disponibilidad.
La memoria se comparte entre tareas y estructuras del executor; no se divide automáticamente
en cuotas iguales por tarea. Esta configuración remota no se crea en el ensayo local.

## Datos de partida: recuperar los resultados de 05
`ENTRADA` contiene los resultados originales. `OUT` contiene exclusivamente esta revisión.
No usar el script `spark_taller.py` como si leyera automáticamente estos productos: sus rutas son distintas.

In [6]:
rendimiento = spark.read.parquet(str(ENTRADA / "spark_rendimiento"))
municipios = spark.read.parquet(str(ENTRADA / "dim_municipio.parquet"))
eva = spark.read.parquet(str(ENTRADA / "eva.parquet"))
claves = ["codigo_municipio", "cultivo", "estado_fisico", "anio"]
rendimiento.printSchema()
print("Particiones: EVA", eva.rdd.getNumPartitions(), "agregado", rendimiento.rdd.getNumPartitions())
print("Configuración de shuffle:", spark.conf.get("spark.sql.shuffle.partitions"))
municipios.show(5, truncate=False)

root
 |-- codigo_municipio: string (nullable = true)
 |-- cultivo: string (nullable = true)
 |-- estado_fisico: string (nullable = true)
 |-- anio: long (nullable = true)
 |-- produccion_t: double (nullable = true)
 |-- area_ha: double (nullable = true)
 |-- rendimiento_t_ha: double (nullable = true)

Particiones: EVA 1 agregado 2
Configuración de shuffle: 4


[Stage 21:>                                                         (0 + 1) / 1]

+----------------+-------------------+----------+------------+
|codigo_municipio|codigo_departamento|municipio |departamento|
+----------------+-------------------+----------+------------+
|05001           |05                 |MEDELLÍN  |ANTIOQUIA   |
|05002           |05                 |ABEJORRAL |ANTIOQUIA   |
|05004           |05                 |ABRIAQUÍ  |ANTIOQUIA   |
|05021           |05                 |ALEJANDRÍA|ANTIOQUIA   |
|05030           |05                 |AMAGÁ     |ANTIOQUIA   |
+----------------+-------------------+----------+------------+
only showing top 5 rows


## 1. Comprobar la dimensión antes del join
Una dimensión duplicada multiplicaría filas del resultado. Spark no ofrece `validate="many_to_one"` en `join`; construir explícitamente las comprobaciones. Un código no encontrado se reporta, no se elimina.

In [7]:
duplicados = municipios.groupBy("codigo_municipio").count().filter(F.col("count") > 1)
assert duplicados.count() == 0, "Revisar duplicados de la dimensión antes de unir"
assert municipios.filter(F.col("codigo_municipio").isNull()).count() == 0
sin_dimension = rendimiento.join(municipios.select("codigo_municipio"), "codigo_municipio", "left_anti")
sin_dimension.select("codigo_municipio").distinct().show(truncate=False)
sin_dimension.write.mode("overwrite").parquet(str(OUT / "revision_sin_dimension"))
filas_antes = rendimiento.count()
print("Grupos antes:", filas_antes, "sin dimensión:", sin_dimension.count())

+----------------+
|codigo_municipio|
+----------------+
+----------------+



Grupos antes: 107620 sin dimensión: 0


## 2. LEFT JOIN normal y broadcast explícito
Comparar planes y conservar el número de grupos. Spark puede elegir broadcast automáticamente con una dimensión pequeña; observar el plan real en lugar de presuponer una estrategia. El hint es razonable solo si la dimensión cabe en cada executor; aquí se trabaja en una única máquina.

In [8]:
union_normal = rendimiento.join(municipios, "codigo_municipio", "left")
union_broadcast = rendimiento.join(F.broadcast(municipios), "codigo_municipio", "left")
union_normal.explain("formatted")
union_broadcast.explain("formatted")
assert union_normal.count() == filas_antes
assert union_broadcast.count() == filas_antes

== Physical Plan ==
AdaptiveSparkPlan (7)
+- Project (6)
   +- BroadcastHashJoin LeftOuter BuildRight (5)
      :- Scan parquet  (1)
      +- BroadcastExchange (4)
         +- Filter (3)
            +- Scan parquet  (2)


(1) Scan parquet 
Output [7]: [codigo_municipio#264, cultivo#265, estado_fisico#266, anio#267L, produccion_t#268, area_ha#269, rendimiento_t_ha#270]
Batched: true
Location: InMemoryFileIndex [file:/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07/spark_rendimiento]
ReadSchema: struct<codigo_municipio:string,cultivo:string,estado_fisico:string,anio:bigint,produccion_t:double,area_ha:double,rendimiento_t_ha:double>

(2) Scan parquet 
Output [4]: [codigo_municipio#271, codigo_departamento#272, municipio#273, departamento#274]
Batched: true
Location: InMemoryFileIndex [file:/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07/dim_municipio.parquet]
PushedFilters: [IsNotNull(codigo_municipio)]
ReadSchema: struct<codigo_municipio:string,codigo_dep

### Medir la materialización del join
Incluye lectura y escritura; el plan se construyó antes. `count()` por sí solo no representa el coste de producir todas las columnas. No se compara rendimiento entre estrategias con una sola vuelta.

In [9]:
%%time
union_broadcast.write.mode("overwrite").parquet(str(OUT / "rendimiento_municipal"))

CPU times: user 2.17 ms, sys: 4.7 ms, total: 6.87 ms
Wall time: 2.35 s


In [10]:
normal_pd = union_normal.toPandas().sort_values(claves).reset_index(drop=True)
union_pd = pd.read_parquet(OUT / "rendimiento_municipal").sort_values(claves).reset_index(drop=True)
pd.testing.assert_frame_equal(normal_pd[union_pd.columns], union_pd, check_dtype=False)
display(union_pd.head())
# Solo el resultado agregado pequeño se lleva a pandas para verificar.

,codigo_municipio,cultivo,estado_fisico,anio,produccion_t,area_ha,rendimiento_t_ha,codigo_departamento,municipio,departamento
0,05001,Aguacate,En fresco,2019,138.00,23.00,6.000000,05,MEDELLÍN,ANTIOQUIA
1,05001,Aguacate,En fresco,2020,114.00,19.00,6.000000,05,MEDELLÍN,ANTIOQUIA
2,05001,Aguacate,En fresco,2021,117.12,19.52,6.000000,05,MEDELLÍN,ANTIOQUIA
3,05001,Aguacate,En fresco,2022,141.12,23.52,6.000000,05,MEDELLÍN,ANTIOQUIA
4,05001,Aguacate,En fresco,2023,128.12,19.52,6.563525,05,MEDELLÍN,ANTIOQUIA


## 3. Ventana: tres municipios por cultivo, estado y año
`row_number` asigna posiciones sin colapsar filas. Particionar por cultivo, estado y año, ordenar rendimiento descendente y desempatar por código municipal ascendente. El desempate permite reproducir el top 3; no implica superioridad agronómica ni causalidad.

In [11]:
from pyspark.sql.window import Window
ventana = Window.partitionBy("cultivo", "estado_fisico", "anio").orderBy(
    F.col("rendimiento_t_ha").desc(), F.col("codigo_municipio").asc())
ranking = rendimiento.withColumn("posicion", F.row_number().over(ventana))
top3 = ranking.filter(F.col("posicion") <= 3)
top3.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Filter (8)
   +- Window (7)
      +- WindowGroupLimit (6)
         +- Sort (5)
            +- Exchange (4)
               +- WindowGroupLimit (3)
                  +- Sort (2)
                     +- Scan parquet  (1)


(1) Scan parquet 
Output [7]: [codigo_municipio#264, cultivo#265, estado_fisico#266, anio#267L, produccion_t#268, area_ha#269, rendimiento_t_ha#270]
Batched: true
Location: InMemoryFileIndex [file:/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07/spark_rendimiento]
ReadSchema: struct<codigo_municipio:string,cultivo:string,estado_fisico:string,anio:bigint,produccion_t:double,area_ha:double,rendimiento_t_ha:double>

(2) Sort
Input [7]: [codigo_municipio#264, cultivo#265, estado_fisico#266, anio#267L, produccion_t#268, area_ha#269, rendimiento_t_ha#270]
Arguments: [cultivo#265 ASC NULLS FIRST, estado_fisico#266 ASC NULLS FIRST, anio#267L ASC NULLS FIRST, rendimiento_t_ha#270 DESC NULLS LAST, codigo_municipio#264 A

In [12]:
%%time
top3.write.mode("overwrite").parquet(str(OUT / "spark_top3"))

CPU times: user 3.38 ms, sys: 7.32 ms, total: 10.7 ms
Wall time: 4.41 s


## 4. Verificar el ranking con SQL en DuckDB
Para aislar la semántica de la ventana, ambos motores leen el mismo resultado agregado Spark. La equivalencia numérica del agregado contra DuckDB ya se verificó en 05. Evitamos que diferencias de coma flotante mínimas alteren un empate aparente.

In [13]:
con = duckdb.connect()
con.read_parquet(str(ENTRADA / "spark_rendimiento/*.parquet")).create_view("rendimiento")
sql_top3 = """
SELECT * FROM (
  SELECT *, ROW_NUMBER() OVER (
    PARTITION BY cultivo, estado_fisico, anio
    ORDER BY rendimiento_t_ha DESC, codigo_municipio ASC
  ) AS posicion
  FROM rendimiento
) WHERE posicion <= 3
"""
top_duckdb = con.sql(sql_top3).df()
top_spark = pd.read_parquet(OUT / "spark_top3")
orden = ["cultivo", "estado_fisico", "anio", "posicion"]
pd.testing.assert_frame_equal(
    top_duckdb.sort_values(orden).reset_index(drop=True)[top_spark.columns],
    top_spark.sort_values(orden).reset_index(drop=True), check_dtype=False)
assert top_spark.groupby(["cultivo", "estado_fisico", "anio"]).size().le(3).all()
top_spark.sort_values(orden).to_csv(OUT / "top3_verificado.csv", index=False)
display(top_spark.sort_values(orden).head(12))
con.close()

,codigo_municipio,cultivo,estado_fisico,anio,produccion_t,area_ha,rendimiento_t_ha,posicion
0,05440,Acelga,En fresco,2019,185.50,7.00,26.500000,1
1,25175,Acelga,En fresco,2019,500.00,20.00,25.000000,2
2,25473,Acelga,En fresco,2019,714.00,34.00,21.000000,3
3,05440,Acelga,En fresco,2020,265.00,10.00,26.500000,1
4,25473,Acelga,En fresco,2020,693.00,33.00,21.000000,2
5,25214,Acelga,En fresco,2020,253.20,12.66,20.000000,3
6,05440,Acelga,En fresco,2021,238.50,9.00,26.500000,1
7,25473,Acelga,En fresco,2021,1560.00,70.00,22.285714,2
8,68147,Acelga,En fresco,2021,60.00,3.00,20.000000,3
9,05440,Acelga,En fresco,2022,198.75,7.50,26.500000,1


## 5. Agregación, particiones y AQE
La consulta mantiene las claves y el cociente de sumas de 05. Se compara su resultado completo,
no solo el conteo. Este experimento no cambia núcleos, número de máquinas ni tamaño del dataset.

**Ensayo A:** cambiar de cuatro a dos particiones con AQE activo, como en el notebook original.
AQE puede combinar particiones; los archivos de salida no tienen por qué ser cuatro o dos.

**Ensayo B:** desactivar AQE de forma temporal para aislar la configuración del shuffle.
Esto no es una recomendación de rendimiento. Al terminar se restauran los valores anteriores.
`Window.partitionBy` define grupos de comparación; no configura máquinas ni directorios.

In [14]:
evidencia = []
config_antes = {k: spark.conf.get(k) for k in ["spark.sql.shuffle.partitions", "spark.sql.adaptive.enabled"]}
try:
    for aqe in [True, False]:
        spark.conf.set("spark.sql.adaptive.enabled", str(aqe).lower())
        for numero in [4, 2]:
            spark.conf.set("spark.sql.shuffle.partitions", str(numero))
            consulta = (eva.filter(F.col("apto_rendimiento")).groupBy(*claves)
                .agg(F.sum("produccion_t").alias("produccion_t"), F.sum("area_ha").alias("area_ha"))
                .withColumn("rendimiento_t_ha", F.col("produccion_t") / F.col("area_ha")))
            etiqueta = f"aqe_{str(aqe).lower()}_shuffle_{numero}"
            plan = guardar_plan(consulta, f"{etiqueta}_plan_inicial.txt")
            # Los nombres shuffle_4 y shuffle_2 indican escenarios, no archivos internos del shuffle.
            destino = OUT / ("aqe_activo" if aqe else "aqe_inactivo") / f"shuffle_{numero}"
            sc.setJobGroup("REV01_" + etiqueta, etiqueta)
            try:
                inicio = time.perf_counter()
                consulta.write.mode("overwrite").parquet(str(destino))
                segundos = time.perf_counter() - inicio
            finally:
                sc.setLocalProperty("spark.jobGroup.id", None)
                sc.setLocalProperty("spark.job.description", None)
            recuperado = pd.read_parquet(destino).sort_values(claves).reset_index(drop=True)
            original_pd = pd.read_parquet(ENTRADA / "spark_rendimiento").sort_values(claves).reset_index(drop=True)
            pd.testing.assert_frame_equal(original_pd[recuperado.columns], recuperado,
                                          check_dtype=False, atol=1e-6, rtol=0)
            evidencia.append({"aqe": aqe, "shuffle_configurado": numero,
                              "archivos_parquet": len(list(destino.glob("part-*.parquet"))),
                              "grupos": len(recuperado), "equivalente": True,
                              "segundos_lectura_calculo_escritura": segundos, "salida": str(destino)})
finally:
    for clave_config, valor in config_antes.items():
        spark.conf.set(clave_config, valor)
display(pd.DataFrame(evidencia))
pd.DataFrame(evidencia).to_csv(OUT / "evidencias/experimento_particiones.csv", index=False)

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Project (7)
   +- HashAggregate (6)
      +- Exchange (5)
         +- HashAggregate (4)
            +- Project (3)
               +- Filter (2)
                  +- Scan parquet  (1)


(1) Scan parquet 
Output [7]: [codigo_municipio#275, anio#277L, cultivo#278, estado_fisico#280, area_ha#281, produccion_t#282, apto_rendimiento#289]
Batched: true
Location: InMemoryFileIndex [file:/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07/eva.parquet]
PushedFilters: [IsNotNull(apto_rendimiento), EqualTo(apto_rendimiento,true)]
ReadSchema: struct<codigo_municipio:string,anio:bigint,cultivo:string,estado_fisico:string,area_ha:double,produccion_t:double,apto_rendimiento:boolean>

(2) Filter
Input [7]: [codigo_municipio#275, anio#277L, cultivo#278, estado_fisico#280, area_ha#281, produccion_t#282, apto_rendimiento#289]
Condition : (isnotnull(apto_rendimiento#289) AND apto_rendimiento#289)

(3) Project
Output [6]: [codigo_municipio#275, anio#

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Project (7)
   +- HashAggregate (6)
      +- Exchange (5)
         +- HashAggregate (4)
            +- Project (3)
               +- Filter (2)
                  +- Scan parquet  (1)


(1) Scan parquet 
Output [7]: [codigo_municipio#275, anio#277L, cultivo#278, estado_fisico#280, area_ha#281, produccion_t#282, apto_rendimiento#289]
Batched: true
Location: InMemoryFileIndex [file:/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07/eva.parquet]
PushedFilters: [IsNotNull(apto_rendimiento), EqualTo(apto_rendimiento,true)]
ReadSchema: struct<codigo_municipio:string,anio:bigint,cultivo:string,estado_fisico:string,area_ha:double,produccion_t:double,apto_rendimiento:boolean>

(2) Filter
Input [7]: [codigo_municipio#275, anio#277L, cultivo#278, estado_fisico#280, area_ha#281, produccion_t#282, apto_rendimiento#289]
Condition : (isnotnull(apto_rendimiento#289) AND apto_rendimiento#289)

(3) Project
Output [6]: [codigo_municipio#275, anio#

== Physical Plan ==
* Project (8)
+- * HashAggregate (7)
   +- Exchange (6)
      +- * HashAggregate (5)
         +- * Project (4)
            +- * Filter (3)
               +- * ColumnarToRow (2)
                  +- Scan parquet  (1)


(1) Scan parquet 
Output [7]: [codigo_municipio#275, anio#277L, cultivo#278, estado_fisico#280, area_ha#281, produccion_t#282, apto_rendimiento#289]
Batched: true
Location: InMemoryFileIndex [file:/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07/eva.parquet]
PushedFilters: [IsNotNull(apto_rendimiento), EqualTo(apto_rendimiento,true)]
ReadSchema: struct<codigo_municipio:string,anio:bigint,cultivo:string,estado_fisico:string,area_ha:double,produccion_t:double,apto_rendimiento:boolean>

(2) ColumnarToRow [codegen id : 1]
Input [7]: [codigo_municipio#275, anio#277L, cultivo#278, estado_fisico#280, area_ha#281, produccion_t#282, apto_rendimiento#289]

(3) Filter [codegen id : 1]
Input [7]: [codigo_municipio#275, anio#277L, cultivo#278, estad

== Physical Plan ==
* Project (8)
+- * HashAggregate (7)
   +- Exchange (6)
      +- * HashAggregate (5)
         +- * Project (4)
            +- * Filter (3)
               +- * ColumnarToRow (2)
                  +- Scan parquet  (1)


(1) Scan parquet 
Output [7]: [codigo_municipio#275, anio#277L, cultivo#278, estado_fisico#280, area_ha#281, produccion_t#282, apto_rendimiento#289]
Batched: true
Location: InMemoryFileIndex [file:/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07/eva.parquet]
PushedFilters: [IsNotNull(apto_rendimiento), EqualTo(apto_rendimiento,true)]
ReadSchema: struct<codigo_municipio:string,anio:bigint,cultivo:string,estado_fisico:string,area_ha:double,produccion_t:double,apto_rendimiento:boolean>

(2) ColumnarToRow [codegen id : 1]
Input [7]: [codigo_municipio#275, anio#277L, cultivo#278, estado_fisico#280, area_ha#281, produccion_t#282, apto_rendimiento#289]

(3) Filter [codegen id : 1]
Input [7]: [codigo_municipio#275, anio#277L, cultivo#278, estad

,aqe,shuffle_configurado,archivos_parquet,grupos,equivalente,segundos_lectura_calculo_escritura,salida
0,True,4,2,107620,True,4.174455,/mnt/c/Users/TUPTC/bigdata/big-data-postgradua...
1,True,2,2,107620,True,3.578019,/mnt/c/Users/TUPTC/bigdata/big-data-postgradua...
2,False,4,4,107620,True,4.832324,/mnt/c/Users/TUPTC/bigdata/big-data-postgradua...
3,False,2,2,107620,True,3.953633,/mnt/c/Users/TUPTC/bigdata/big-data-postgradua...


Una escritura crea su propia ejecución SQL. El `explain` anterior es el plan de consulta antes de
materializar la escritura: **no se presenta como el plan adaptativo final de esa escritura**.
Consultar la ejecución de escritura en SQL/Stages de la UI o sus event logs para comprobar el plan real.
En el ejemplo pequeño del bloque A, `collect()` sí materializa la propia consulta inspeccionada.

Comparar tiempos aquí sirve para describir estas ejecuciones. Una sola vuelta, con diferente
calentamiento y caché del sistema, no demuestra que una configuración sea más rápida.

## 6. Dónde quedaron los archivos y qué se escribió
Volver a inventariar **antes de detener Spark**. Los bloques de shuffle suelen incluir archivos
`shuffle_*.data`, `shuffle_*.index` y, según configuración, checksums. Otros temporales pueden usar
prefijos `temp_shuffle_` o `temp_local_`. El nombre de un temporal no demuestra por sí solo spill.

| Ubicación | Contenido |
|---|---|
| Directorios efectivos `blockmgr-*` | Bloques y temporales de trabajo de Spark |
| `OUT/aqe_activo/shuffle_4` y otros escenarios | Resultados finales Parquet de nuestras escrituras |
| `OUT/event_logs` | Eventos de ejecución; no son archivos de datos del shuffle |
| `OUT/evidencias` | Inventarios, planes y métricas capturados por el notebook |

Los temporales pueden eliminarse antes de esta inspección. El tamaño presente es una fotografía,
no el total histórico de bytes escritos. No se borran ni alteran manualmente archivos internos.

In [15]:
inventario_final = inventario("antes_stop")
inicial_por_ruta = {f["ruta"]: f for f in inventario_inicial}
final_por_ruta = {f["ruta"]: f for f in inventario_final}
comparacion_archivos = {
    "nuevos_presentes": sorted(set(final_por_ruta) - set(inicial_por_ruta)),
    "ya_no_presentes": sorted(set(inicial_por_ruta) - set(final_por_ruta)),
    "cambiados_presentes": sorted(p for p in set(final_por_ruta) & set(inicial_por_ruta)
                                 if final_por_ruta[p] != inicial_por_ruta[p]),
}
guardar_json("comparacion_inventarios.json", comparacion_archivos)
print({k: len(v) for k, v in comparacion_archivos.items()})
print("Directorios temporales efectivos:", [str(p) for p in directorios_reales])
print("Salidas Parquet:", [e["salida"] for e in evidencia])

antes_stop archivos presentes: 45 bytes presentes: 2337240


,ruta,bytes,modificado_ns
0,/mnt/c/Users/TUPTC/bigdata/big-data-postgradua...,40,1792274785206389700
1,/mnt/c/Users/TUPTC/bigdata/big-data-postgradua...,40,1792274786301367500
2,/mnt/c/Users/TUPTC/bigdata/big-data-postgradua...,40,1792274786401764700
3,/mnt/c/Users/TUPTC/bigdata/big-data-postgradua...,156,1792274785199192200
4,/mnt/c/Users/TUPTC/bigdata/big-data-postgradua...,40,1792274786466719700
5,/mnt/c/Users/TUPTC/bigdata/big-data-postgradua...,168,1792274782218694900
6,/mnt/c/Users/TUPTC/bigdata/big-data-postgradua...,65,1792274786219108000
7,/mnt/c/Users/TUPTC/bigdata/big-data-postgradua...,68637,1792274807630553000
8,/mnt/c/Users/TUPTC/bigdata/big-data-postgradua...,40,1792274786607452400
9,/mnt/c/Users/TUPTC/bigdata/big-data-postgradua...,69,1792274786370782500


{'nuevos_presentes': 45, 'ya_no_presentes': 0, 'cambiados_presentes': 0}
Directorios temporales efectivos: ['/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07_rev01/temporales_spark/blockmgr-4e75f611-e93a-49c5-8ce4-abe50d5393d1']
Salidas Parquet: ['/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07_rev01/aqe_activo/shuffle_4', '/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07_rev01/aqe_activo/shuffle_2', '/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07_rev01/aqe_inactivo/shuffle_4', '/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07_rev01/aqe_inactivo/shuffle_2']


## 7. Shuffle Write y spill: medir sin presuponer
- **Shuffle Write:** bytes escritos para el intercambio entre etapas.
- **Spill (Disk):** bytes derramados a disco por presión de memoria en operadores compatibles.
- **Spill (Memory):** tamaño estimado en memoria de los datos derramados. No es otra escritura física.

Es posible observar Shuffle Write positivo y spill igual a cero. Cero spill es válido para esta
carga y estos recursos; no significa que no existan escrituras en disco. Si la captura falla,
se registra ausencia de evidencia, no un cero. No se reduce artificialmente la memoria ni se
multiplican los datos agrícolas para forzar un resultado.

En la UI revisar SQL, Stages (tareas y timeline) y Executors. Antes de cerrar Spark, anotar los IDs
de etapa y tareas que respaldan la interpretación. Los event logs pueden revisarse con un History
Server si el entorno dispone de él; este notebook no instala ni inicia ese servicio.

In [16]:
etapas_finales = capturar_metricas("antes_stop")
resumen_disco = None
if etapas_finales is not None:
    completas = [e for e in etapas_finales if e["status"] == "COMPLETE"]
    resumen_disco = {k: sum(e.get(k, 0) for e in completas)
                     for k in ["shuffleWriteBytes", "shuffleReadBytes", "memoryBytesSpilled", "diskBytesSpilled"]}
    print("Acumulado de intentos de etapa completos de esta aplicación:", resumen_disco)
    print("No corresponde a un único join ni al tamaño actual de los directorios.")
    print("Spill a disco observado:", resumen_disco["diskBytesSpilled"] > 0)
    guardar_json("resumen_disco.json", resumen_disco)
else:
    print("Evidencia incompleta: revisar UI y event logs; no informar spill=0.")

,stageId,attemptId,status,numTasks,numCompleteTasks,shuffleReadBytes,shuffleWriteBytes,memoryBytesSpilled,diskBytesSpilled
0,66,0,COMPLETE,2,2,2197441,0,0,0
1,65,0,COMPLETE,1,1,0,2197441,0,0
2,64,0,COMPLETE,4,4,2473384,0,0,0
3,63,0,COMPLETE,1,1,0,2473384,0,0
4,62,0,COMPLETE,2,2,2197441,0,0,0
...,...,...,...,...,...,...,...,...,...
62,4,0,COMPLETE,2,2,0,118,0,0
63,3,0,SKIPPED,2,0,0,0,0,0
64,2,0,COMPLETE,2,2,341,0,0,0
65,1,0,SKIPPED,2,0,0,0,0,0


Acumulado de intentos de etapa completos de esta aplicación: {'shuffleWriteBytes': 9488725, 'shuffleReadBytes': 9488725, 'memoryBytesSpilled': 0, 'diskBytesSpilled': 0}
No corresponde a un único join ni al tamaño actual de los directorios.
Spill a disco observado: False


## 8. Interpretación y control de entrega
Completar respuestas con IDs, archivos y valores observados. No inventar métricas.
Si un mecanismo no se observó, decirlo y explicar el límite del ensayo.

In [17]:
respuestas = {
    "particion_tarea_executor": "PENDIENTE: distinguir las tres unidades y citar el ensayo B.",
    "concurrencia": "PENDIENTE: indicar executor, tareas y solapamiento observado; no confundirlo con CPU utilizada.",
    "agregacion_parcial": "PENDIENTE: relacionar la reconstrucción didáctica con los operadores del plan.",
    "ubicacion_shuffle": "PENDIENTE: citar directorios efectivos y distinguirlos de las salidas Parquet.",
    "spill": "PENDIENTE: indicar métricas reales, o ausencia de evidencia; cero es válido.",
    "aqe": "PENDIENTE: comparar configuración, etapas y archivos sin equipararlos a máquinas.",
    "limite_escalabilidad": "PENDIENTE: explicar el límite de la preparación Pandas de 05 y de local[2].",
}
control = {"revision": "01", "spark": spark.version, "master": sc.master,
    "application_id": sc.applicationId, "ui": sc.uiWebUrl,
    "entrada": str(ENTRADA), "salida": str(OUT),
    "directorios_temporales_efectivos": [str(p) for p in directorios_reales],
    "event_logs": str(OUT / "event_logs"),
    "grupos_antes": filas_antes, "grupos_despues": len(union_pd),
    "grupos_sin_dimension": sin_dimension.count(), "filas_top3": len(top_spark),
    "ranking_equivalente": True, "particiones": evidencia,
    "concurrencia_observada": concurrencia_observada, "metricas_disco": resumen_disco,
    "respuestas": respuestas}
(OUT / "control_joins_ventanas_rev01.json").write_text(
    json.dumps(control, ensure_ascii=False, indent=2), encoding="utf-8")
display(control)

{'revision': '01',
 'spark': '4.0.1',
 'master': 'local[2]',
 'application_id': 'local-1792274733986',
 'ui': 'http://10.255.255.254:4040',
 'entrada': '/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07',
 'salida': '/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07_rev01',
 'directorios_temporales_efectivos': ['/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07_rev01/temporales_spark/blockmgr-4e75f611-e93a-49c5-8ce4-abe50d5393d1'],
 'event_logs': '/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07_rev01/event_logs',
 'grupos_antes': 107620,
 'grupos_despues': 107620,
 'grupos_sin_dimension': 0,
 'filas_top3': 2990,
 'ranking_equivalente': True,
 'particiones': [{'aqe': True,
   'shuffle_configurado': 4,
   'archivos_parquet': 2,
   'grupos': 107620,
   'equivalente': True,
   'segundos_lectura_calculo_escritura': 4.174455016007414,
   'salida': '/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07_rev01/aqe_activo/s

## 9. Cierre explícito
Antes de ejecutar la celda siguiente, terminar la inspección de la UI y completar las respuestas.
`spark.stop()` puede limpiar los temporales y cierra la UI activa. Los inventarios guardados y
los event logs permanecen como evidencia. Tras detener Spark, no ejecutar celdas de consulta
sin comenzar de nuevo desde la configuración.

In [18]:
spark.stop()
print("Sesión detenida. Evidencias:", OUT / "evidencias")
print("Event logs conservados:")
for p in sorted((OUT / "event_logs").rglob("*")):
    if p.is_file():
        print(p, p.stat().st_size, "bytes")

Sesión detenida. Evidencias: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07_rev01/evidencias
Event logs conservados:
/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07_rev01/event_logs/local-1792246349374 592578 bytes
/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07_rev01/event_logs/local-1792267330061 112687 bytes
/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07_rev01/event_logs/local-1792274733986 2340622 bytes


## Referencias y límites
- [Spark 4.0.1: arquitectura](https://spark.apache.org/docs/4.0.1/cluster-overview.html).
- [Configuración y recursos](https://spark.apache.org/docs/4.0.1/configuration.html).
- [Planes, particiones y AQE](https://spark.apache.org/docs/4.0.1/sql-performance-tuning.html).
- [Spark UI, API REST, métricas y event logs](https://spark.apache.org/docs/4.0.1/monitoring.html).
- [DiskBlockManager 4.0.1: directorios y archivos](https://github.com/apache/spark/blob/v4.0.1/core/src/main/scala/org/apache/spark/storage/DiskBlockManager.scala).
- [SortShuffleManager 4.0.1: escritura y recuperación de bloques](https://github.com/apache/spark/blob/v4.0.1/core/src/main/scala/org/apache/spark/shuffle/sort/SortShuffleManager.scala).

No se prueba pérdida de executors, recuperación del driver, un clúster remoto ni rendimiento a TB.
Esos mecanismos requieren otro entorno y otro diseño experimental. Se preserva el significado
agrícola: cociente de sumas, claves, cardinalidad y ranking reproducible.